In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

In [1]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy(operational=True, min_num_qubits=16).name

> **Note:** Ang mga Qiskit Function ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ang mga ito at maaaring magbago.

## Pangkalahatang-ideya
Sa quantum chemistry, ang electronic structure problem ay nakatuon sa paghahanap ng mga solusyon sa electronic Schrödinger equation — ang mga quantum wave function na naglalarawan ng gawi ng mga elektron ng sistema. Ang mga wave function na ito ay mga vector ng mga complex amplitude, kung saan ang bawat amplitude ay kumakatawan sa kontribusyon ng isang posibleng electron configuration.

Ang ground state ay ang pinakamababang energy wave function ng sistema at may espesyal na kahalagahan sa pag-aaral ng mga molecular system. Ang pinaka-tumpak na paraan para kalkulahin ang ground state ay isinasaalang-alang ang lahat ng posibleng electron configuration, ngunit ito ay nagiging imposible para sa mas malalaking sistema dahil ang bilang ng mga configuration ay lumalaki nang eksponensyal habang lumalaki ang sistema.

Ang Handover Iterative Variational Quantum Eigensolver (HI-VQE) ay isang makabagong hybrid quantum-classical na pamamaraan para sa tumpak na pagtatantya ng ground state ng mga molecular system. Pinagsasama nito ang quantum hardware at classical computing, gumagamit ng mga quantum processor para mahusay na tuklasin ang mga kandidatong electron configuration at kinakalkula ang nagresultang wave function sa mga classical computer. Sa pamamagitan ng pagbuo ng mga compact ngunit chemically accurate na wave function, pinapabuti ng HI-VQE ang pananaliksik at pagtuklas sa quantum chemistry at materials science.

![Larawan na nagpapakita ng pangkalahatang-ideya ng HI-VQE algorithm ng Qunova](../docs/images/guides/qunova-chemistry/overview.svg)

Binabawasan ng HI-VQE ang computational complexity ng electronic structure problem sa pamamagitan ng mahusay na pagtatantya ng ground state nang may mataas na katumpakan. Nakatuon ito sa isang maingat na piniling subset ng mga pinaka-kaugnay na electron configuration, ino-optimize ang parehong katumpakan at kahusayan.

Sa pagsasama ng mga kalakasan ng parehong classical at quantum computer, paulit-ulit na pinipino at pinapabuti ng HI-VQE ang kasalukuyang tinatantyang wave function. Ang natatanging mga teknik sa pagbuo ng subspace nito ay tumutulong na gawing mas mahusay ang pagpili ng configuration, para ang mga gumagamit ay may mas malaking computational control at pinahusay na katumpakan sa mga quantum chemistry simulation.

Kung gusto mong matuto nang mas malalim tungkol sa algorithm, maaari kang [basahin ang kaugnay na research paper](https://arxiv.org/abs/2503.06292).

## Paglalarawan
Ang bilang ng mga electron configuration para sa isang molecular system ay lumalaki nang eksponensyal habang lumalaki ang sistema. Gayunpaman, para sa ilang partikular na electronic state, tulad ng ground state, karaniwan na isang maliit na bahagi lamang ng mga configuration ang may malaking kontribusyon sa energy ng estado. Sinasamantala ng mga Selected Configuration Interaction (SCI) na pamamaraan ang paguusok na ito para mabawasan ang mga gastos sa computation sa pamamagitan ng pagtukoy at pagtuon sa mga pinaka-kaugnay na configuration. Ang subset ng mga configuration na ito ay tinatawag na subspace.

Ginagamit ng HI-VQE ang likas na kahusayan ng mga quantum computer sa pagrepresenta ng mga molecular system para tulungan ang paghahanap ng subspace. Pinagsasama nito ang mga classical at quantum subroutine para malutas ang electronic structure problem nang may mataas na katumpakan. Hindi tulad ng mga umiiral na quantum SCI na pamamaraan, pinagsasama ng HI-VQE ang variational training, iterative subspace construction, at pre-diagonalization configuration screening para mapahusay ang kahusayan sa pamamagitan ng pagbabawas ng mga quantum measurement, iteration, at classical diagonalization cost. Kaya naman, maaaring ilapat ang HI-VQE sa mas malalaking molecular system na nangangailangan ng mas maraming qubit, at binabawasan ang gastos para malutas ang isang problemang may partikular na laki sa parehong antas ng katumpakan.

![Larawan na nagpapakita ng detalyadong paglalarawan ng bawat hakbang sa HI-VQE algorithm ng Qunova.](../docs/images/guides/qunova-chemistry/description.avif)

Para kalkulahin ang ground state ng isang sistema, unang ginagamit ng HI-VQE ang classical chemistry package na PySCF para bumuo ng molecular representation mula sa mga input na ibinigay ng gumagamit, tulad ng molecular geometry at iba pang impormasyon tungkol sa molekula. Pagkatapos ay pumapasok ito sa isang hybrid quantum-classical optimization loop, paulit-ulit na pinipino ang isang subspace para optimal na marepresenta ang ground state habang pinipigilan ang bilang ng mga kasama sa configuration. Nagpapatuloy ang loop hanggang matugunan ang mga convergence criteria, tulad ng subspace size o energy stability, pagkatapos ay ini-output ang computed ground state wave function at energy. Maaaring gamitin ang mga resultang ito para bumuo ng tumpak na potential energy surface at magsagawa ng karagdagang pagsusuri ng sistema.

Ang optimization loop ay nakatuon sa pag-aayos ng mga parameter ng quantum Circuit para makabuo ng mataas na kalidad na subspace. Nag-aalok ang HI-VQE ng tatlong opsyon sa quantum Circuit: [`excitation_preserving`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.excitation_preserving), [efficient_su2](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2), at [LUCJ](https://qiskit-community.github.io/ffsim/explanations/lucj.html). Ang optimization ay nagsisimula malapit sa Hartree-Fock reference state dahil sa pangkalahatang angkop nito. Ang Circuit ay isinasagawa pagkatapos sa isang quantum device at ang mga configuration ay sinasample mula sa nagresultang quantum state bago ibalik bilang binary string. Dahil sa ingay ng quantum device, ang ilang sampled na configuration ay maaaring hindi pisikal na wasto, nabigo ang pag-conserve ng electron number o spin. Tinutugunan ito ng HI-VQE gamit ang configuration recovery process mula sa package na [qiskit-addon-sqd](/guides/qiskit-addons-sqd#sample-based-quantum-diagonalization-sqd-overview), para ang mga gumagamit ay makakapag-correct ng mga hindi wastong configuration o itatapon ang mga ito.

Ang mga wastong configuration ay dumadaan sa isang opsyonal na screening step para alisin ang mga inaasahang minimal ang kontribusyon. Binabawasan nito ang dimensyon ng subspace, kaya naman mas mababang gastos ang diagonalization step. Kung pinagana ang screening, isang paunang subspace Hamiltonian ang binubuo mula sa mga wastong configuration at isang diagonalization ang isinasagawa na may maluwag na termination criteria. Kahit mababa ang katumpakan ng mga nagresultang amplitude para sa bawat configuration, ito ay epektibo para hulaan kung aling mga configuration ang dapat iwanan sa labas ng subspace sa iteration na ito, at mabilis itong kalkulahin.

Ang mga piniling configuration ay idadagdag sa subspace, at ang Hamiltonian ng sistema ay ini-project sa subspace na ito. Ang subspace ay paulit-ulit na naa-update, pinapanatili ang mga pinaka-kaugnay na configuration sa buong mga iteration. Ang diskarteng ito ay kaibahan sa mga alternatibong pamamaraan dahil ang quantum Circuit ay hindi kailangang tantiyahin ang buong ground state sa bawat hakbang.

Susunod, ang subspace Hamiltonian ay classically diagonalized para makuha ang pinakamababang eigenvalue at ang kaukulang eigenvector nito, na kumakatawan sa isang approximation ng ground state at ng energy nito. Habang nag-iimprove ang kalidad ng subspace sa pamamagitan ng mga iteration, mas malapitan ng computed ground state ang tunay na ground state. Isang karagdagang screening step ay maaaring isagawa sa puntong ito para alisin ang anumang configuration mula sa subspace na walang malaking kontribusyon sa computed ground state. Tinitiyak ng hakbang na ito na ang subspace na dadalhin sa susunod na iteration ay kasing-compact ng posible. Sinusuri ito batay sa mga amplitude na ibinalik ng diagonalization, dahil kinakatawan ng mga ito ang kontribusyon ng bawat configuration sa computed ground state.

Pagkatapos, tinutukoy ng convergence check kung ang karagdagang training ay magpapabuti ng mga resulta. Kung oo, isang opsyonal na classical expansion step ang isasagawa, ia-update ang mga parameter ng quantum Circuit para higit pang mabawasan ang computed energy, at uulitin ang proseso. Ang classical expansion step ay nagbibigay ng karagdagang mga configuration para sa subspace, dinadagdagan ang mga configuration na sinample mula sa quantum device. Una nitong tinutukoy ang configuration na may pinakamalaking amplitude sa mga resulta ng diagonalization, bago bumuo ng mga bagong configuration na may single at double excitation mula sa natukoy na configuration. Ang desired na bilang ng mga configuration na ito ay idadagdag sa subspace.

Kapag natukoy na ang mga iteration ay nag-converge na, ibinabalik ng HI-VQE ang computed ground state (sa anyo ng mga estado sa subspace at ang kanilang mga amplitude sa ground state wave function), ang energy nito, at isang energy variance measure na nagbibigay ng indikasyon kung ang computed state ay bumubuo ng isang eigenstate ng Hamiltonian ng sistema.

Maaaring magdesisyon ang mga gumagamit ng quantum Circuit na gagamitin at ang bilang ng mga shot na kukunin para sa bawat quantum Circuit, pati na rin kontrolin ang subspace size o paganahin ang classical generation ng karagdagang configuration para tulungan ang mga configuration na galing sa quantum. Sa ganitong paraan, maaaring i-customize ng mga gumagamit ang gawi ng HI-VQE ayon sa kanilang nais na mga aplikasyon.

## Pagsisimula
Una, [humiling ng access sa function](https://forms.office.com/r/zN3hvMTqJ1).
Pagkatapos, i-authenticate gamit ang iyong [IBM Quantum&reg; API key](http://quantum.cloud.ibm.com/) at, kung nai-save na nito ang iyong [account](/guides/functions#install-qiskit-functions-catalog-client) sa iyong lokal na environment, piliin ang Qiskit Function tulad ng sumusunod:

In [ ]:
import reprlib
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("qunova/hivqe-chemistry")

## Mga Input
Tingnan ang sumusunod na talahanayan para sa lahat ng input parameter na tinatanggap ng function. Ang mga sumusunod na seksyon ng [Mga opsyon sa molekula](#molecule-options) at [Mga opsyon ng HI-VQE](#hi-vqe-options) ay nagbibigay ng mas detalyadong impormasyon tungkol sa mga argumentong iyon.
| Pangalan               | Uri                                                            | Paglalarawan                                                                                                                                                                                                                                                                                                | Kinakailangan | Default                                                                  | Halimbawa                                                                                 |
|------------------------|----------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| geometry               | Union[List[List[Union[str, Tuple[float, float, float]]]], str] | Maaaring maging string o structured list na naglalaman ng mga pares ng atom at coordinate. Kung ito ay ibinibigay bilang string, dapat ito ay molecular geometry sa Cartesian Coordinate Format. Kung ito ay ibinibigay bilang list, dapat itong ibigay bilang list ng mga list na bawat isa ay naglalaman ng atom string at coordinate tuple. | Oo      | N/A                                                                      | `[['O', (0, 0, 0)], ['H', (0, 1, 0)], ['H', (0, 0, 1)]]` o `"O 0 0 0; H 0 1 0; H 0 0 1"` |
| backend\_name          | str                                                            | Pangalan ng backend para sa query.                                                                                                                                                                                                                                                                          | Oo      | N/A                                                                      | `ibm_fez`                                                                                 |
| max\_states            | int                                                            | Ang maximum na dimensyon ng subspace para sa diagonalization. Mas kaunting estado ang gagamitin kung ang bilang ay hindi perfect square.                                                                                                                                                                     | Oo      | N/A                                                                      | `100`                                                                                     |
| max\_expansion\_states | int                                                            | Ang maximum na bilang ng classically-generated na CI state na isasama sa bawat iteration.                                                                                                                                                                                                                   | Oo      | N/A                                                                      | `10`                                                                                      |
| molecule_options                | dict                                                           | Mga opsyon na may kaugnayan sa molekulang ginagamit bilang input sa HI-VQE. Tingnan ang seksyong [Mga opsyon sa molekula](#molecule-options) para sa mas detalyadong impormasyon.                                                                                                                            | Hindi       | Tingnan ang seksyong [Mga opsyon sa molekula](#molecule-options) para sa mga detalye.  | `{"basis": "sto3g", "unit": "angstrom" }`                               |
| hivqe_options                | dict                                                           | Mga opsyon na nagkokontrol sa gawi ng HI-VQE algorithm. Tingnan ang seksyong [Mga opsyon ng HI-VQE](#hi-vqe-options) para sa mas detalyadong impormasyon.                                                                                                                                                   | Hindi       | Tingnan ang seksyong [Mga opsyon ng HI-VQE](#hi-vqe-options) para sa mga detalye.  | `{"shots": 10_000, "max_iter": 10 }`                               |
### Mga opsyon sa molekula
Ang sumusunod na talahanayan ay nagdedetalye ng lahat ng key at value na maaaring itakda sa `molecule_options` dictionary, pati na rin ang kanilang mga uri ng data at default na value. Lahat ng key ay opsyonal.

| Key                               | Uri ng value                        | Default na Value                 | Wastong saklaw                                                                                                                                                 | Paliwanag                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"charge"`                        | `int`                               | `0`                              | Iba't iba                                                                                                                                                      | Isang integer na nagtatakda ng kabuuang net charge ng molecular system. Ang default na value ay 0; ngunit maaari itong maging anumang integer.                                                                                                                                                                                                                                                                                                                                                                                          |
| `"basis"`                         | `str`                               | `'sto-3g'`                       | Iba't iba                                                                                                                                                      | Isang string na nagtatakda ng uri ng basis; ipinipasa ang mga ito sa pyscf. (hal: `"sto-3g"`, `"3-21g"`, `"6-31g"`, `"cc-pvdz"`)                                                                                                                                                                                                                                                                                                                                                                                                       |
| `"active_orbitals"`               | `List[int]`                         | Bawat orbital index.             | Ang spatial orbital index na wasto para sa problema.                                                                                                           | Isang list ng mga active orbital index sa interval na [0, n) kung saan ang n ay ang bilang ng qubit na ginagamit sa problema. Kung ito ay tinukoy, dapat ding tukuyin ang frozen_orbitals argument.                                                                                                                                                                                                                                                                                                                                     |
| `"frozen_orbitals"`               | `List[int]`                         | Walang index.                    | Ang spatial orbital index na wasto para sa problema, hindi kasama ang mga active orbital.                                                                      | Isang list ng mga frozen orbital index sa parehong saklaw ng active_orbitals. Kung tinukoy, dapat ding tukuyin ang active_orbitals. Tandaan na ang mga occupied orbital lamang ang dapat i-freeze dahil ang bilang ng mga active electron ay nababawasan ng 2 para sa bawat occupied orbital na na-freeze.                                                                                                                                                                                                                               |
| `"orbital_coeffs"`                | `List[List[float]]`                 | Hartree-Fock molecular orbital. | Iba't iba.                                                                                                                                                     | Ang mga coefficient para sa mga spatial orbital na ginagamit sa pagkalkula ng electronic repulsion integral para sa sistema. Ilang wastong halimbawa ay ang mga Hartree-Fock molecular orbital, natural orbital, at AVAS orbital.                                                                                                                                                                                                                                                                                                        |
| `"symmetry"`                      | `Union[str, bool]`                  | `False`                          | `True` o `False`                                                                                                                                               | Ginagamit para i-invoke ang point group symmetry para sa mga paunang molecular calculation para mabuo ang symmetry adapted orbital basis. Ginagamit ang mga symmetry-adapted orbital na ito bilang mga basis function para sa mga sumusunod na SCF calculation. Ang default na value ay False; kung itinakda sa True, ito ay ivo-invoke at ang mga arbitrary na point group ay awtomatikong matatukoy at gagamitin. Kung isang partikular na symmetry ang itinalaga, halimbawa, symmetry = "Dooh", magtatataas ng error kung ang molecular geometry ay hindi napapailalim sa kinakailangang symmetry na ito. |
| `"symmetry_subgroup"`             | `Optional[str]`                     | `None`                           | Tingnan ang [dokumentasyon ng pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                          | Maaaring gamitin para bumuo ng subgroup ng natukoy na symmetry. Walang epekto ito kapag ang symmetry ay tinukoy gamit ang symmetry keyword argument.                                                                                                                                                                                                                                                                                                                                                                                     |
| `"unit"`                          | `str`                               | `"angstrom"`                     | Tingnan ang [dokumentasyon ng pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                          | Nagtatakda ng unit ng sukat na gagamitin para sa mga atomic coordinate at distansya. Ang default ay gumamit ng mga angstrom unit.                                                                                                                                                                                                                                                                                                                                                                                                        |
| `"nucmod"`                        | `Optional[Union[dict, str]]`        | `None`                           | Tingnan ang [dokumentasyon ng pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                          | Nagtatakda ng nuclear model na gagamitin. Bilang default, gumagamit ito ng point nuclear model, at ang iba pang value ay nagbibigay-daan sa Gaussian nuclear model. Kung isang function ang ibinibigay, gagamitin ito kasama ang Gaussian nuclear model para makabuo ng nuclear charge distribution value na 'zeta'.                                                                                                                                                                                                                       |
| `"pseudo"`                        | `Optional[Union[dict, str]]`        | `None`                           | Tingnan ang [dokumentasyon ng pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                          | Nagtatakda ng pseudopotential para sa mga atom sa molekula. Ito ay None bilang default, na nagpapahiwatig na walang pseudopotential ang inilalapat at lahat ng elektron ay tahasang kasama sa mga kalkulasyon.                                                                                                                                                                                                                                                                                                                           |
| `"cart"`                          | `bool`                              | `False`                          | Tingnan ang [dokumentasyon ng pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                          | Nagtatakda kung gagamit ng cartesian GTO bilang mga angular momentum basis function sa kalkulasyon. Ang default na value ng False ay gagamit ng spherical GTO.                                                                                                                                                                                                                                                                                                                                                                          |
| `"magmom"`                        | `Optional[List[Union[int, float]]]` | `None`                           | Tingnan ang [dokumentasyon ng pyscf](https://pyscf.org/pyscf_api_docs/pyscf.gto.html#pyscf.gto.mole.MoleBase.build).                                          | Itinatakda ang colinear spin magnetic moment ng bawat atom. Bilang default, ito ay None at ang bawat atom ay nagsisimula sa spin na zero.                                                                                                                                                                                                                                                                                                                                                                                                |
| `"avas_aolabels"`                 | `Optional[List[str]]`               | `None`                           | hal. ["H 1s", "O 2p"] para sa H2O                                                                                                                                                         | Tinutukoy nito ang Atomic Orbital na isasama sa AVAS scheme. Tingnan ang [dokumentasyon ng AVAS](https://arxiv.org/abs/1701.07862).                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"avas_threshold"`                | `float`                             | `0.2`                            | Sa pagitan ng 0.0 at 2.0                                                                                                                                                    | Tinutukoy nito ang cutoff value na ginagamit para matukoy kung aling Atomic Orbital (AO) ang mananatili sa active space.                                                                                                                                                                                                                                                                                                                                                                                                                |
| `"noons_level"`                   | `Optional[str]`                     | `None`                           | `"mp2"` o `"ccsd"`                                                                                                                                             | Tinutukoy nito ang theoretical na diskarte para sa paghahanda ng mga natural orbital at pagpili ng mga active orbital batay sa Natural Orbital Occupation Numbers (NOONs) scheme. Tingnan ang [dokumentasyon ng NOONs](https://doi.org/10.1063/5.0042147). Ang parehong active at frozen orbital index ay dapat ibigay para makontrol ang bilang ng mga orbital (at ang bilang ng mga qubit).                                                                                                                                              |
### Mga opsyon ng HI-VQE
Ang sumusunod na talahanayan ay nagdedetalye ng lahat ng key at value na maaaring itakda sa `hivqe_options` dictionary, pati na rin ang kanilang mga uri ng data at default na value. Lahat ng key ay opsyonal.

| Key                               | Uri ng value                        | Default na Value                 | Wastong saklaw                                                                                                                                                 | Paliwanag                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
|:----------------------------------|:-----------------------------------:|:--------------------------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `"shots"`                         | `int`                               | `1_000`                          | Sa pagitan ng 1 at 10 000.                                                                                                                                     | Bilang ng mga shot na gagamitin sa quantum device bawat iteration.                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| `"max_iter"`                      | `int`                               | `25`                             | Sa pagitan ng 1 at 50.                                                                                                                                         | Ang maximum na bilang ng iteration para i-optimize ang ansatz. Maaaring gumamit ang algorithm ng mas kaunting iteration kung ang convergence ay maabot nang maaga.                                                                                                                                                                                                                                                                                                                                                                       |
| `"initial_basis_states"`          | `List[str]`                         | Ang Hartree-Fock state.          | Mga binary string na may bilang ng bits na tumutugma sa kinakailangang bilang ng qubit para sa problema.                                                        | Maaaring gamitin para i-restart ang algorithm gamit ang mga classical state mula sa nakaraang resulta.                                                                                                                                                                                                                                                                                                                                                                                                                                  |
| `"ansatz"`                        | `str`                               | `"epa"`                          | `"epa"`, `"hea"`, o `"lucj"`                                                                                                                                   | Tinutukoy nito ang quantum ansatz na io-optimize para makabuo ng mga bagong estado. `"epa"` ay pumipili ng excitation-preserving ansatz. `"hea"` ay pumipili ng hardware-efficient ansatz. `"lucj"` ay pumipili ng local unitary cluster Jastrow ansatz.                                                                                                                                                                                                                                                                                  |
| `"convergence_count"`             | `int`                               | `3`                              | Hindi bababa sa 2.                                                                                                                                             | Ang bilang ng mga iteration na walang malaking pagbabago sa computed energy na dapat lumipas bago ituring na nag-converge na ang algorithm.                                                                                                                                                                                                                                                                                                                                                                                              |
| `"convergence_abstol"`            | `float`                             | `1e-4`                           | Higit sa 0 at hindi hihigit sa 1.                                                                                                                              | Ang magnitude ng pagbabago sa computed energy na itinuturing na malaki para sa mga layunin ng convergence check.                                                                                                                                                                                                                                                                                                                                                                                                                        |
| `"reset_convergence_count"`       | `bool`                              | `True`                           | `True` o `False`                                                                                                                                               | Kung `True`, ang mga iteration ng `convergence_count` ay dapat lumipas nang walang mapanggalong malaking pagbabago para maging kwalipikado bilang nag-converge. Kung `False`, ang algorithm ay titigil pagkatapos ng `convergence_count` kung ang mga hindi malaking pagbabago ay naganap sa anumang iteration sa panahon ng optimization process.                                                                                                                                                                                          |
| `"configuration_recovery"`        | `bool`                              | `True`                           | `True` o `False`.                                                                                                                                              | Kung gagamitin o hindi ang configuration recovery mula sa package na `qiskit-addon-sqd`. Kung True, ang mga hindi wastong estado na sinample mula sa quantum device ay naitatama nang classically. Kung False, itinatatapon ang mga ito.                                                                                                                                                                                                                                                                                                 |
| `"ansatz_entanglement"`           | `str`                               | `"circular"`                     | Alinman sa `"linear"`, `"reverse_linear"`, `"pairwise"`, `"circular"`, `"full"`, o `"sca"`. Kung gumagamit ng `"lucj"` ansatz, ang `"lucj_default"` ay opsyon din. | Tinutukoy nito ang entanglement scheme na gagamitin sa loob ng quantum Circuit, sumusunod sa mga karaniwang Qiskit at [ffsim convention para sa LUCJ ansatz](https://qiskit-community.github.io/ffsim/how-to-guides/simulate-lucj.html).                                                                                                                                                                                          |
| `"ansatz_reps"`                   | `int`                               | `2`                              | Higit sa 0.                                                                                                                                                    | Ang bilang ng pag-uulit ng bawat layer sa quantum Circuit.                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
| `"amplitude_screening_tolerance"` | `Union[float,int]`                  | `0`                              | Hindi bababa sa 0, at mas mababa sa 1.                                                                                                                         | Ang tolerance para sa pagpapasya kung aling mga estado ang dapat i-screen out ng subspace pagkatapos ng diagonalization. Tinutukoy nito ang inclusion threshold para sa mga estado ng subspace batay sa kanilang mga computed amplitude.                                                                                                                                                                                                                                                                                                 |
| `"overlap_screening_tolerance"`   | `float`                             | `1e-2`                           | Sa pagitan ng `1e-4` at `1e-1`, kasama ang mga dulo.                                                                                                           | Ang tolerance para sa paghula kung aling mga estado ang dapat i-screen out ng subspace bago ang diagonalization. Kinokontrol nito ang katumpakan ng mga hinulaang amplitude para sa bawat estado, kung saan ang mas maliit na value ay nagresulta sa mas tumpak na mga hula.                                                                                                                                                                                                                                                             |
## Mga Output
Nagbabalik ang function ng dictionary na may apat na key at value. Ang mga key at value ay naidokumento sa sumusunod na talahanayan:

| Key             | Uri ng Value  | Paliwanag                                                                                                               |
|:----------------|---------------|---------------------------------------------------------------------------------------------------------------------------|
| `"energy"`      | `float`       | Ang tinatayang ground state energy ng molekula.                                                                          |
| `"states"`      | `List[str]`   | Ang mga piniling determinant na bumubuo ng subspace na ginamit para malutas ang energy. Nasa alternating alpha-beta format ang mga ito. |
| `"eigenvector"` | `List[float]` | Ang eigenvector na tumutugma sa ground state ng subspace na binubuo ng `"states"`.                                       |
| `"energy_variance"` | `float` | Ang energy variance ng ground state ng subspace na binubuo ng `"states"`, na nagbibigay ng indikasyon ng kalidad ng solusyon. Ang value na ito ay hindi negatibo at ang mas mababang value ay nangangahulugang mas malapit ang ground state ng subspace sa isang eigenstate ng Hamiltonian ng sistema. |
| `"energy_history"` | `List[float]` | Ang mga energy na kinakalkula sa bawat iteration sa panahon ng hybrid optimization process, sa parehong pagkakasunud-sunod na kinakalkula ang mga ito. Dalawang energy ang kinakalkula bawat iteration bilang bahagi ng SPSA optimization process. |
## Halimbawa
Ang unang halimbawa ay nagpapakita kung paano kalkulahin ang ground state energy para sa isang NH3 na molekula gamit ang HI-VQE algorithm.
#### Tukuyin ang molecular geometry at mga opsyon
Ang molecular geometry ng NH3 ay ibinibigay na may mga Cartesian coordinate na pinaghihiwalay ng ";" para sa bawat atom.

In [3]:
# Define the molecule geometry
geometry = """
N         -0.85188       -0.02741        0.03141;
H          0.16545        0.00593       -0.01648;
H         -1.16348       -0.39357       -0.86702;
H         -1.16348        0.94228        0.06281;
"""

Maaaring tukuyin at ibigay ang karagdagang mga opsyon para sa molecular system sa sumusunod na format ng dictionary.

In [4]:
# Configure some options for the job.
molecule_options = {"basis": "sto3g"}
hivqe_options = {"shots": 100, "max_iter": 20}

Isagawa ang function na may geometry at mga input na opsyon.

In [5]:
# Run HI-VQE
job = function.run(
    geometry=geometry,
    # `backend_name` is the name of a backend with at least 16 qubits, for example, "ibm_marrakesh".
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=10,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

Magandang ideya na i-print ang Function job ID para maibigay ito sa mga support request kung may magkamaling mangyari.

In [6]:
print("Job ID:", job.job_id)

Job ID: 128ee399-7cfc-4be2-91e9-c4abd22b97c7


This example then utilizes 16 qubits with 8 orbitals of sto3g basis for an NH3 molecule.

Check your Qiskit Function workload's [status](/docs/guides/functions#check-job-status) or return [results](/docs/guides/functions#retrieve-results) as follows:

In [7]:
print(job.status())

QUEUED


Ang halimbawang ito ay gumagamit ng 16 qubit na may 8 orbital ng sto3g basis para sa isang NH3 na molekula.
Suriin ang [status](/guides/functions#check-job-status) ng iyong Qiskit Function workload o kunin ang mga [resulta](/guides/functions#retrieve-results) tulad ng sumusunod:

In [8]:
result = job.result()

# Output can be long, so we display a shortened representation
shortened_result = reprlib.repr(result)
print(shortened_result)

{'eigenvector': [0.9824200343205695, 0.009520846167419264, 6.365368845740147e-08, 3.6832123006425615e-07, 0.0012869941182066654, 2.3403891050875465e-05, ...], 'energy': -55.521146537970466, 'energy_history': [-55.52091922342852, -55.52113695367561, -55.521146537970466, -55.52114653864798, -55.521146537970466, -55.521146537970466, ...], 'energy_variance': 9.788555455342562e-12, ...}


To access the ground state energy, use the "energy" key. The "eigenvector" key provides the CI coefficients with corresponding bitstring notation of electron configuration stored with "states" of the results.

In [10]:
fci_energy = -55.521148034704126  # the exact energy using FCI method
hivqe_energy = result["energy"]
print(
    f"|Exact Energy - HI-VQE Energy|: {abs(fci_energy - hivqe_energy) * 1000} mHa"
)
print(f"Sampled Number of States: {len(result['states'])}")

|Exact Energy - HI-VQE Energy|: 0.0014967336596782843 mHa
Sampled Number of States: 1936


Pagkatapos makumpleto ang job, maaaring makuha ang mga resulta gamit ang `result()` instance.

In [11]:
# This cell is hidden from users
backend_name = service.least_busy(operational=True, min_num_qubits=38).name

In [12]:
# Define Li2S geometries
Li2S_geoms = {
    "Li2S_1.51": "S        -1.239044    0.671232   -0.030374;Li       -1.506327    0.432403   -1.498949;Li       -0.899996    0.973348    1.826768;",
    "Li2S_2.40": "S        -1.741432    0.680397    0.346702;Li       -0.529307    0.488006   -1.729343;Li       -1.284307    0.989409    2.177209;",
    "Li2S_3.80": "S        -2.707255    0.674298    0.909161;Li        0.079218    0.552012   -1.671656;Li       -0.927010    0.931502    1.557063;",
}

# Configure some options for the job.
molecule_options = {
    "basis": "sto3g",
}
hivqe_options = {
    "shots": 100,
    "max_iter": 20,
}

results = []
for geom in ["Li2S_1.51", "Li2S_2.40", "Li2S_3.80"]:
    # Run HI-VQE
    job = function.run(
        geometry=Li2S_geoms[geom],
        backend_name=backend_name,  # can use any device with at least 38 qubits
        max_states=2000,
        max_expansion_states=10,
        molecule_options=molecule_options,
        hivqe_options=hivqe_options,
    )
    results.append(job.result())

Para ma-access ang ground state energy, gamitin ang "energy" key. Ang "eigenvector" key ay nagbibigay ng mga CI coefficient na may kaukulang bitstring notation ng electron configuration na nakaimbak sa "states" ng mga resulta.

In [13]:
job = function.run(
    geometry="invalid-geometry",  # This will cause an error
    backend_name=backend_name,
    max_states=2000,
    max_expansion_states=15,
    molecule_options=molecule_options,
    hivqe_options=hivqe_options,
)

job.result()

QiskitServerlessException: {"message": "An unexpected error occurred during job execution. Please make sure that your inputs are valid. If you are still experiencing problems, you can contact the Qunova Computing support service at qiskit.support@qunovacomputing.com and provide the Function job ID of this job for more assistance.", "status": "failure"}

In [14]:
job.status()

'ERROR'

Output:

|Exact Energy - HI-VQE Energy|: 0.077237504 mHa
Sampled Number of States: 1936
## Pagganap
Ang seksyong ito ay nagpapakita ng mga ipinahayag na benchmark calculation ng HI-VQE na may 24-qubit na kaso para sa Li2S, isang 40-qubit na kaso para sa isang N2 na molekula, at isang 44-qubit na kaso para sa isang FeP-NO system.
#### Dissociation potential energy surface curve para sa isang Li2S na molekula na may 24 qubit
Ang PES curve ay ipinapakita kasama ang FCI reference at paunang hula mula sa RHF, kasama ang energy error mula sa FCI reference.

![Larawan na nagpapakita na ang HI-VQE ay gumagawa ng mga solusyon sa loob ng chemical accuracy ng classical reference PES curve para sa Li2S system.](../docs/images/guides/qunova-chemistry/Li2S_PES.avif)

Ang mga kalkulasyon ay isinagawa gamit ang mga sumusunod na geometry at opsyon.